# Основы синтеза речи №2

##  План

1. Encodec
2. Conditional TTS
3. Voice Cloning

## EnCodec

https://github.com/facebookresearch/encodec

In [4]:
# ! pip install encodec datasets transformers==4.57.3

In [5]:
import encodec
import torch
import torchaudio

In [6]:
from datasets import load_dataset, Audio
from transformers import EncodecModel, AutoProcessor

In [7]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [8]:
model = EncodecModel.from_pretrained("facebook/encodec_24khz").eval().to(DEVICE)
processor = AutoProcessor.from_pretrained("facebook/encodec_24khz")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/809 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/93.1M [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [9]:
wav, sr = torchaudio.load("https://ruslan-corpus.github.io/audio/01.wav")

In [11]:
sr, processor.sampling_rate

(44100, 24000)

In [12]:
audio_sample = torchaudio.functional.resample(wav, sr, processor.sampling_rate)

In [13]:
inputs = processor(raw_audio=audio_sample, sampling_rate=processor.sampling_rate, return_tensors="pt")

ValueError: Expected mono audio but example has 1 channels

In [14]:
inputs = processor(raw_audio=audio_sample[0], sampling_rate=processor.sampling_rate, return_tensors="pt")

In [18]:
(inputs["padding_mask"] == 1.0).all()

tensor(True)

In [19]:
inputs

{'padding_mask': tensor([[1, 1, 1,  ..., 1, 1, 1]], dtype=torch.int32), 'input_values': tensor([[[ 0.0312,  0.0675,  0.0835,  ..., -0.0312, -0.0370, -0.0236]]])}

In [ ]:
inputs["input_values"].shape

torch.Size([1, 1, 94080])

In [29]:
with torch.no_grad():
  encoder_outputs = model.encode(inputs["input_values"].to(DEVICE), inputs["padding_mask"].to(DEVICE), bandwidth=1.5)
  decoder_outputs = model.decode(encoder_outputs.audio_codes.to(DEVICE), encoder_outputs.audio_scales, inputs["padding_mask"].to(DEVICE))

In [30]:
encoder_outputs.audio_codes.shape

torch.Size([1, 1, 2, 294])

In [31]:
encoder_outputs.audio_codes.shape

torch.Size([1, 1, 2, 294])

In [32]:
decoder_outputs.audio_values.shape

torch.Size([1, 1, 94080])

In [34]:
94080 / (2 * 294)

160.0

In [35]:
from IPython.display import display, Audio

In [36]:
display(Audio(inputs["input_values"][0], rate=processor.sampling_rate))

In [38]:
display(Audio(decoder_outputs.audio_values.cpu()[0], rate=processor.sampling_rate))

In [39]:
model = encodec.EncodecModel.encodec_model_24khz()

/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Downloading: "https://dl.fbaipublicfiles.com/encodec/v0/encodec_24khz-d7cc33bc.th" to /root/.cache/torch/hub/checkpoints/encodec_24khz-d7cc33bc.th


100%|██████████| 88.9M/88.9M [00:00<00:00, 117MB/s]


In [41]:
# The number of codebooks used will be determined by the bandwidth selected.
# E.g. for a bandwidth of 6kbps, `n_q = 8` codebooks are used.
# Supported bandwidths are 1.5kbps (n_q = 2), 3 kbps (n_q = 4), 6 kbps (n_q = 8) and 12 kbps (n_q =16) and 24kbps (n_q=32).
# For the 48 kHz model, only 3, 6, 12, and 24 kbps are supported. The number
# of codebooks for each is half that of the 24 kHz model as the frame rate is twice as much.
model.set_target_bandwidth(24)

In [42]:
# Load and pre-process the audio waveform
wav, sr = torchaudio.load("https://ruslan-corpus.github.io/audio/01.wav")
wav = encodec.utils.convert_audio(wav, sr, model.sample_rate, model.channels)
wav = wav.unsqueeze(0)

In [43]:
wav.shape

torch.Size([1, 1, 94080])

In [44]:
with torch.no_grad():
  encoded = model.encode(wav)

In [46]:
codes, scale = encoded[0]

In [48]:
codes.shape, scale

(torch.Size([1, 32, 294]), None)

In [49]:
wav.shape[-1] / (codes.shape[-1] * codes.shape[-2])

10.0

In [54]:
codes

tensor([[[ 465,  251,  479,  ...,  499,   91,  393],
         [ 493,  711,  628,  ...,  213,  364,  898],
         [1017,  598,  722,  ...,  674,  187,  458],
         ...,
         [ 953,  330,  403,  ..., 1022,  573,  171],
         [ 899,  617,  540,  ...,  640,  319,  322],
         [ 628,  760,  968,  ...,  990,  642,  817]]])

In [51]:
with torch.no_grad():
  decoded = model.decode([(codes, scale)])

In [52]:
decoded.shape

torch.Size([1, 1, 94080])

In [53]:
display(Audio(decoded[0], rate=processor.sampling_rate))

In [56]:
data = encodec.compress(model, wav[0])

In [57]:
len(data)

11827

In [58]:
wav.dtype

torch.float32

In [59]:
(wav.shape[-1] * wav.dtype.itemsize) / len(data)

31.818719878244693

In [60]:
(wav.shape[-1] * 2) / len(data)

15.909359939122346

## Qwen3-TTS

![image.png](https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen3-TTS-Repo/qwen3_tts_introduction.png)

![image](https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen3-TTS-Repo/overview.png)

### Установка зависимостей

1. `qwen-tts` - основная библиотека
    - https://github.com/QwenLM/Qwen3-TTS
2. Flash Attention - быстрая реализация механизма внимания
    - пакет `flash-attn` - может собираться очень долго
    - https://github.com/mjun0812/flash-attention-prebuild-wheels/releases

In [ ]:
!python --version

Python 3.12.12


In [ ]:
import torch
torch.__version__

'2.10.0+cu128'

In [62]:
!pip install qwen-tts sox
# !pip install https://github.com/mjun0812/flash-attention-prebuild-wheels/releases/download/v0.7.16/flash_attn-2.8.3+cu128torch2.10-cp312-cp312-linux_x86_64.whl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.4/61.4 kB 5.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 6.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.5/113.5 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 97.8 MB/s eta 0:00:00
  Created wheel for sox: filename=sox-1.5.0-py3-none-any.whl size=40036 sha256=3b63356ab16084bc42efec0e3d36a51ec3b79b929f4977cffed3313f79505668
  Stored in directory: /root/.cache/pip/wheels/8c/c7/e7/baea1f7e79b9eb53addc81cc9b827424f4a7d8c9cc18c03659
Successfully built sox
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.13.0
    Uninstalling accelerate-1.13.0:
      Successfully uninstalled accelerate-1.13.0


In [2]:
import torch
from qwen_tts import Qwen3TTSModel

### Используем предобученные голоса


| Speaker | Voice Description  |  Native language |
| --- | --- | --- |
| Vivian | Bright, slightly edgy young female voice. | Chinese |
| Serena | Warm, gentle young female voice. | Chinese |
| Uncle_Fu | Seasoned male voice with a low, mellow timbre. | Chinese |
| Dylan | Youthful Beijing male voice with a clear, natural timbre. | Chinese (Beijing Dialect) |
| Eric | Lively Chengdu male voice with a slightly husky brightness. | Chinese (Sichuan Dialect) |
| Ryan | Dynamic male voice with strong rhythmic drive. | English |
| Aiden | Sunny American male voice with a clear midrange. | English |
| Ono_Anna | Playful Japanese female voice with a light, nimble timbre. | Japanese |
| Sohee | Warm Korean female voice with rich emotion. | Korean |

In [3]:
model = Qwen3TTSModel.from_pretrained(
    "Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice",
    device_map="cuda:0",
    dtype=torch.bfloat16,
    # attn_implementation="flash_attention_2",
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.83G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/245 [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

...

speech_tokenizer/model.safetensors:   0%|          | 0.00/682M [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/127 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

In [5]:
wavs, sr = model.generate_custom_voice(
    text="Генерируем речь с заданным голосом",
    language="Russian",
    speaker="Ryan",
    instruct="",
)

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


In [6]:
wavs[0].shape, sr

((72960,), 24000)

In [8]:
from IPython.display import display, Audio

In [9]:
display(Audio(wavs[0], rate=sr))

In [10]:
wavs, sr = model.generate_custom_voice(
    text="Генерируем речь с заданным голосом",
    language="Russian",
    speaker="Ryan",
    instruct="быстро",
)
display(Audio(wavs[0], rate=sr))

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


### Моделирование голоса (voice design)

In [11]:
model = Qwen3TTSModel.from_pretrained(
    "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign",
    device_map="cuda:0",
    dtype=torch.bfloat16,
    # attn_implementation="flash_attention_2",
)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.83G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/245 [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

config.json: 0.00B [00:00, ?B/s]

...

speech_tokenizer/model.safetensors:   0%|          | 0.00/682M [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/127 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

In [12]:
wavs, sr = model.generate_voice_design(
    text=[
      "It's in the top drawer... wait, it's empty? No way, that's impossible! I'm sure I put it there!"
    ],
    language=["English"],
    instruct=[
      "Speak in an incredulous tone, but with a hint of panic beginning to creep into your voice."
    ]
)

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


In [13]:
display(Audio(wavs[0], rate=sr))

In [14]:
wavs, sr = model.generate_voice_design(
    text=[
      "Пробуем генерировать речь, моделируя особенности речи через текстовый промпт",
      "Пробуем генерировать речь, моделируя особенности речи через текстовый промпт",
    ],
    language=["Russian", "Russian"],
    instruct=[
      "Говори громко, отчетливо, проговаривая каждую букву",
      "Говори шепотом, тихо и слаборазборчиво",
    ]
)

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


In [15]:
display(Audio(wavs[0], rate=sr))

In [ ]:
display(Audio(wavs[1], rate=sr))

### Клонирование голоса

In [16]:
model = Qwen3TTSModel.from_pretrained(
    "Qwen/Qwen3-TTS-12Hz-1.7B-Base",
    device_map="cuda:0",
    dtype=torch.bfloat16,
    # attn_implementation="flash_attention_2",
)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/245 [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

...

speech_tokenizer/model.safetensors:   0%|          | 0.00/682M [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/127 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

In [17]:
wavs, sr = model.generate_voice_clone(
    text="Киты — морские млекопитающие из инфраотряда китообразных, не относящиеся ни к дельфинам, ни к морским свиньям.",
    language="Russian",
    ref_audio="https://ruslan-corpus.github.io/audio/01.wav",
    x_vector_only_mode=True,
)

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


In [18]:
display(Audio(wavs[0], rate=sr))

In [ ]:
(wav, sr)

In [21]:
wavs, sr = model.generate_voice_clone(
    text="Киты — морские млекопитающие из инфраотряда китообразных, не относящиеся ни к дельфинам, ни к морским свиньям.",
    language="Russian",
    voice_clone_prompt=
    # ref_audio="https://ruslan-corpus.github.io/audio/01.wav",
    # ref_text="Это было в октябре тысяча девятьсот сорок пятого года.",
)

ValueError: ref_text is required when x_vector_only_mode=False (ICL mode). Bad index=0

In [20]:
display(Audio(wavs[0], rate=sr))

In [22]:
model.create_voice_clone_prompt?